In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()

# Clone or Update

In [ ]:
!git clone https://github.com/magnimel/jersey-number-pipeline.git

In [ ]:
import os
os.chdir('/content/jersey-number-pipeline/')

In [ ]:
# !git switch fahd-lstm

In [ ]:
# RUN THIS CELL TO UPDATE YOUR CODE AFTER YOU COMMIT LOCALLY
!git fetch
!git pull

# Setup

In [ ]:
!pip install gdown

In [ ]:
!python ./setup.py dataset

In [ ]:
!conda run -n jersey --no-capture-output python downloadables.py

# Run Code Below

### (Optional) Subset configuration
Edit `SUBSET_FRACTION` below to run on a random slice of the data. `1.0` = full run.

In [ ]:
SUBSET_FRACTION = 0.0001
RANDOM_SEED     = 42

if SUBSET_FRACTION < 1.0:
    import os, random, json
    import configuration as cfg

    random.seed(RANDOM_SEED)
    split = 'train'
    images_root = os.path.join(
        cfg.dataset['SoccerNet']['root_dir'],
        cfg.dataset['SoccerNet'][split]['images']
    )
    all_images = []
    for tracklet in os.listdir(images_root):
        track_dir = os.path.join(images_root, tracklet)
        if not os.path.isdir(track_dir): continue
        for f in os.listdir(track_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(track_dir, f))

    n = max(1, int(len(all_images) * SUBSET_FRACTION))
    subset = random.sample(all_images, n)
    print(f'Subset: {n} / {len(all_images)} images ({SUBSET_FRACTION*100:.1f}%)')

    # Write a filtered legible JSON that main.py will pick up via --subset flag
    subset_file = os.path.join(
        cfg.dataset['SoccerNet']['working_dir'], 'subset_images.json'
    )
    with open(subset_file, 'w') as fh:
        json.dump(subset, fh)
    print(f'Subset image list -> {subset_file}')
    SUBSET_ARG = f'--subset {subset_file}'
else:
    SUBSET_ARG = ''
    print('Running on full dataset.')

### Run pipeline

In [ ]:
!conda run -n jersey --no-capture-output python main.py SoccerNet test --resume --improved

In [ ]:
!python evaluate.py --pred ./out/SoccerNetResults/final_results.json --gt ./data/SoccerNet/test/test_gt.json